In [ ]:
# Setup & Environment
import sys
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

# 1. Path Setup (Point to your 'kaggle-b2' root)
# If on Kaggle: /kaggle/input/your-repo-name/
# Locally: Adjust to your research-notebook/kaggle-b2 path
sys.path.append("../../") 

from shared.utils import seed_everything
from shared.configs.base_config import BaseConfig
from shared.training import DLTrainer, EmbeddingResNet, TitanicDataset

# 2. Config & Discipline Test Initialization
cfg = BaseConfig(
    batch_size=32, 
    epochs=100, 
    learning_rate=1e-3, 
    weight_decay=1e-2,
    artifacts_path="artifacts"
)
seed_everything(cfg.seed)

In [ ]:
# Feature Engineering & Preprocessing
def prepare_data(df):
    # Feature Engineering
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    
    # Missing Values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna('S')
    
    cat_cols = ['Sex', 'Embarked', 'Pclass', 'Title']
    cont_cols = ['Age', 'SibSp', 'Parch', 'Fare']
    
    # Label Encoding for Embeddings
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
        
    return df, cat_cols, cont_cols

# Load Data
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
df, cat_cols, cont_cols = prepare_data(train_df)

# Calculate Embedding Dimensions
emb_dims = [(df[col].nunique(), min(50, (df[col].nunique() + 1) // 2)) for col in cat_cols]

In [ ]:
# The Cross-Validation Loop (Training)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.seed)
X = df[cat_cols + cont_cols]
y = df['Survived']

# Array to store "honest" predictions for the whole training set
all_oof_probs = np.zeros(len(df)) 

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_fold, val_fold = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()
    
    # Scale Numerical Features (Fitted on Train, Applied to Val)
    scaler = StandardScaler()
    train_fold[cont_cols] = scaler.fit_transform(train_fold[cont_cols])
    val_fold[cont_cols] = scaler.transform(val_fold[cont_cols])
    
    # Create DataLoaders
    train_ds = TitanicDataset(train_fold[cat_cols], train_fold[cont_cols], train_fold['Survived'])
    val_ds = TitanicDataset(val_fold[cat_cols], val_fold[cont_cols], val_fold['Survived'])
    
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)
    
    # Initialize Model & Trainer
    model = EmbeddingResNet(emb_dims=emb_dims, n_cont=len(cont_cols))
    trainer = DLTrainer(model, cfg)
    
    # Train and capture the best OOF predictions from the trainer
    fold_oof_probs = trainer.fit(train_loader, val_loader, fold_idx=fold)
    
    # Store in our master OOF array
    all_oof_probs[val_idx] = fold_oof_probs.flatten()
    print(f"✅ Fold {fold} Complete. Best Val Loss: {trainer.best_loss:.4f}")

# Calculate Final CV Score
final_cv_accuracy = accuracy_score(y, (all_oof_probs > 0.5).astype(int))
print(f"\n🔥 FINAL 5-FOLD CV ACCURACY: {final_cv_accuracy:.4f}")

In [ ]:
# Inference and Submission
# 1. Prepare Test Data
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")
test_processed, _, _ = prepare_data(test_df)

# Global Scaler (Fitted on all Training data for best results)
full_scaler = StandardScaler()
full_scaler.fit(df[cont_cols])
test_processed[cont_cols] = full_scaler.transform(test_processed[cont_cols])

test_ds = TitanicDataset(test_processed[cat_cols], test_processed[cont_cols])
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

# 2. Average Predictions across all 5 folds
all_fold_preds = []

for fold in range(5):
    model = EmbeddingResNet(emb_dims=emb_dims, n_cont=len(cont_cols))
    model.load_state_dict(torch.load(f"{cfg.artifacts_path}/model_fold_{fold}.pth"))
    model.to(cfg.device).eval()
    
    fold_preds = []
    with torch.no_grad():
        for x_cat, x_cont in test_loader:
            x_cat, x_cont = x_cat.to(cfg.device), x_cont.to(cfg.device)
            probs = torch.sigmoid(model(x_cat, x_cont))
            fold_preds.append(probs.cpu().numpy())
            
    all_fold_preds.append(np.concatenate(fold_preds))

# Mean across folds and apply 0.5 threshold
final_probs = np.mean(all_fold_preds, axis=0)
final_predictions = (final_probs > 0.5).astype(int).flatten()

# 3. Create Submission
pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": final_predictions
}).to_csv("submission.csv", index=False)

print("🚀 Submission saved! You've cleanly completed the Titanic Redux.")